In [6]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# 1. Load cleaned data
train = pd.read_csv("../data/processed/train_clean.csv")
test = pd.read_csv("../data/processed/test_clean.csv")

# 2. Define base feature lists
# NOTE: We are NOT using "Payment Delay" because it is a data leak.
categorical_cols = ["Contract Length", "Subscription Type", "Gender"]

numeric_cols = [
    "Total Spend",
    "Support Calls",
    "Usage Frequency",
    "Age",
    "Last Interaction",
    "Tenure",
]

# 3. Make sure numeric columns are actually numeric (coerce any junk to NaN)
for df in [train, test]:
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# 4. Optional but helpful feature engineering (no leakage)

# 4a. Spend bin (deciles of Total Spend) – helps tree models
train["SpendBin"] = pd.qcut(
    train["Total Spend"], q=10, labels=False, duplicates="drop"
)
test["SpendBin"] = pd.qcut(
    test["Total Spend"], q=10, labels=False, duplicates="drop"
)

numeric_cols.append("SpendBin")

# 4b. Spend per tenure (normalized spend)
#     Use (Tenure + 1) to avoid division by 0, no leakage.
train["SpendPerTenure"] = train["Total Spend"] / (train["Tenure"] + 1)
test["SpendPerTenure"] = test["Total Spend"] / (test["Tenure"] + 1)

numeric_cols.append("SpendPerTenure")

# 5. Handle categoricals: fill missing with "Unknown"
for df in [train, test]:
    for col in categorical_cols:
        if col in df.columns:
            df[col] = df[col].fillna("Unknown")

# 6. Final feature list (again, no Payment Delay anywhere)
hgb_features = categorical_cols + numeric_cols

X = train[hgb_features].copy()
y = train["Churn"]

# 7. Train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=1234,
    stratify=y,
)

# 8. Preprocessor: one-hot encode categoricals, pass numeric through
hgb_preprocessor = make_column_transformer(
    (OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
    remainder="passthrough"
)

# 9. HistGradientBoostingClassifier model setup
hgb_model = HistGradientBoostingClassifier(
    max_depth=8,          # reasonably deep trees
    learning_rate=0.05,   # smallish learning rate for stability
    max_iter=300,         # number of boosting iterations
    max_leaf_nodes=64,    # control complexity
    l2_regularization=1.0,
    random_state=1234
)

# 10. Pipeline: preprocessing + model
hgb_pipeline = make_pipeline(
    hgb_preprocessor,
    hgb_model
)

# 11. Fit and evaluate on validation set
hgb_fit = hgb_pipeline.fit(X_train, y_train)

val_pred_hgb = hgb_fit.predict_proba(X_val)[:, 1]
auc_hgb = roc_auc_score(y_val, val_pred_hgb)
print("HistGradientBoosting validation AUC:", auc_hgb)


HistGradientBoosting validation AUC: 0.9299577614159124


In [8]:

# 12. Kaggle predictions
X_test = test[hgb_features].copy()
test_pred_hgb = hgb_fit.predict_proba(X_test)[:, 1]

submission_hgb = pd.DataFrame({
    "CustomerID": test["CustomerID"],
    "Churn": test_pred_hgb
})

submission_hgb.to_csv("../data/submissions/base_test.csv", index=False)
print("Saved ../data/submissions/hgb_no_payment_delay.csv")


Saved ../data/submissions/hgb_no_payment_delay.csv


In [10]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# 1. Load cleaned data
train = pd.read_csv("../data/processed/train_clean.csv")
test = pd.read_csv("../data/processed/test_clean.csv")

# 2. Define feature columns
# IMPORTANT: Do NOT include "Payment Delay" (data leak).
categorical_cols = ["Contract Length", "Subscription Type", "Gender"]

numeric_cols = [
    "Total Spend",
    "Support Calls",
    "Usage Frequency",
    "Age",
    "Last Interaction",
    "Tenure",
]

# 3. Make sure numeric columns are numeric
for df in [train, test]:
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# 4. Feature engineering (no leakage)

# 4a. SpendBin: deciles of Total Spend (helps tree models find better splits)
train["SpendBin"] = pd.qcut(
    train["Total Spend"], q=10, labels=False, duplicates="drop"
)
test["SpendBin"] = pd.qcut(
    test["Total Spend"], q=10, labels=False, duplicates="drop"
)
numeric_cols.append("SpendBin")

# 4b. SpendPerTenure: normalized spend (avoid /0 with +1)
train["SpendPerTenure"] = train["Total Spend"] / (train["Tenure"] + 1)
test["SpendPerTenure"] = test["Total Spend"] / (test["Tenure"] + 1)
numeric_cols.append("SpendPerTenure")

# 5. Basic imputation for numeric features (median per column)
for col in numeric_cols:
    median_val = train[col].median()
    train[col] = train[col].fillna(median_val)
    test[col] = test[col].fillna(median_val)

# 6. Handle missing categoricals with "Unknown"
for df in [train, test]:
    for col in categorical_cols:
        if col in df.columns:
            df[col] = df[col].fillna("Unknown")

# 7. Final feature list (still no Payment Delay)
gb_features = categorical_cols + numeric_cols

X = train[gb_features].copy()
y = train["Churn"]

# 8. Train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=1234,
    stratify=y,
)

# 9. Preprocessor: one-hot encode categoricals, pass numeric through
gb_preprocessor = make_column_transformer(
    (OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
    remainder="passthrough"
)

# 10. Gradient Boosting model
gb_model = GradientBoostingClassifier(
    learning_rate=0.05,     # smaller LR + more trees = smoother model
    n_estimators=400,       # number of boosting stages
    max_depth=3,            # depth of individual trees
    min_samples_leaf=100,   # regularization
    subsample=0.8,          # stochastic gradient boosting
    max_features="sqrt",    # feature subsampling
    random_state=1234,
)

# 11. Build pipeline: preprocessing + model
gb_pipeline = make_pipeline(
    gb_preprocessor,
    gb_model
)

# 12. Fit and evaluate on validation set
gb_fit = gb_pipeline.fit(X_train, y_train)

val_pred_gb = gb_fit.predict_proba(X_val)[:, 1]
auc_gb = roc_auc_score(y_val, val_pred_gb)
print("GradientBoosting validation AUC:", auc_gb)

# 13. Kaggle predictions
X_test = test[gb_features].copy()
test_pred_gb = gb_fit.predict_proba(X_test)[:, 1]

submission_gb = pd.DataFrame({
    "CustomerID": test["CustomerID"],
    "Churn": test_pred_gb
})

# submission_gb.to_csv("../data/submissions/gradient_boosting_no_payment_delay.csv", index=False)
print("Saved ../data/submissions/gradient_boosting_no_payment_delay.csv")


GradientBoosting validation AUC: 0.922652053884705
Saved ../data/submissions/gradient_boosting_no_payment_delay.csv
